# DeepGuard : entraînement et analyses sur Kaggle

**Avant de lancer :**
1. *Session options* : accélérateur **GPU** (T4 ou P100) et **Internet On**.
2. *Add Input* : ajoute le dataset **140k Real and Fake Faces** (`xhlulu/140k-real-and-fake-faces`).
   Pour le test de généralisation, ajoute aussi **Real and Fake Face Detection** (`ciplab/real-and-fake-face-detection`).
3. *Add-ons → Secrets* : crée `DAGSHUB_TOKEN` (token DagsHub) pour suivre les runs dans MLflow. Sans lui, les runs restent locaux.
4. Le dépôt GitHub doit être **public** pour le `git clone` ci-dessous (sinon utilise un token).

**Durée indicative sur un T4** : section 4 (trois modèles) ≈ 35 min, section 5 (ViT `scale`) ≈ 12 min,
section 6 (graines, facultative) ≈ 35 min, section 8 (analyses) ≈ 10 min, section 9 (export) ≈ 5 min.
Exécute les cellules dans l'ordre (*Run All* fonctionne ; la section 6 peut être sautée).

In [ ]:
# 1. Récupérer le code et installer le nécessaire (PyTorch, sklearn, pandas, matplotlib existent déjà sur Kaggle)
!git clone https://github.com/sourourbensalha209-alt/deepguard.git
%cd deepguard
!pip install -q timm mlflow pyyaml onnx onnxruntime

In [ ]:
# 2. Chemins des données + suivi MLflow (DagsHub)
import glob, os, yaml

cands = glob.glob("/kaggle/input/**/real-vs-fake", recursive=True)
assert cands, "Dataset introuvable : ajoute 140k Real and Fake Faces via « Add Input »."
os.environ["DATA_DIR"] = cands[0]
print("DATA_DIR =", os.environ["DATA_DIR"])

ext = [p for p in glob.glob("/kaggle/input/**/training_fake", recursive=True)]
EXTERNAL_DIR = os.path.dirname(ext[0]) if ext else None
print("external_dir =", EXTERNAL_DIR)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/sourourbensalha209-alt/deepguard.mlflow"
    os.environ["MLFLOW_TRACKING_USERNAME"] = "sourourbensalha209-alt"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
    print("MLflow : DagsHub")
except Exception as e:
    os.environ.pop("MLFLOW_TRACKING_URI", None)
    print("MLflow : local (secret DAGSHUB_TOKEN absent) :", e)

def set_params(external=None, **train):
    p = yaml.safe_load(open("params.yaml"))
    p["train"].update(train)
    if external:
        p["external_dir"] = external
    yaml.safe_dump(p, open("params.yaml", "w"), sort_keys=False)
    print(p["train"])

if EXTERNAL_DIR:
    set_params(external=EXTERNAL_DIR)

In [ ]:
# 3. Listes de fichiers train / val / test (équilibrées, reproductibles)
!python -m src.data.prepare

## Outil : entraîner + évaluer une expérience et ranger ses résultats
Chaque expérience (`tag` + graine) est entraînée, évaluée (validation + test + second jeu), puis rangée dans
`reports/runs/<tag>_s<graine>/` : `train.json`, `history.json` (courbes), `test.json`, `preds_val.csv`, `preds_test.csv`.
Le checkpoint est copié dans `/kaggle/working/<tag>_s<graine>.pt`. Ces dossiers alimentent toutes les analyses (`src/analysis/`).

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working")
HP = {
    "baseline":     dict(model="baseline",     epochs=10, batch_size=128, lr=1e-3, augment="basic"),
    "efficientnet": dict(model="efficientnet", epochs=3,  batch_size=64,  lr=1e-4, augment="basic"),
    "vit":          dict(model="vit",          epochs=3,  batch_size=32,  lr=3e-5, augment="basic"),
    "vit_scale":    dict(model="vit",          epochs=3,  batch_size=32,  lr=3e-5, augment="scale"),  # recadrages aléatoires
}

def set_seed(seed):
    p = yaml.safe_load(open("params.yaml"))
    p["seed"] = seed  # n'affecte que l'initialisation et l'ordre des lots : data/processed reste identique
    yaml.safe_dump(p, open("params.yaml", "w"), sort_keys=False)

def run(tag, seed=42):
    set_params(**HP[tag])
    set_seed(seed)
    get_ipython().system("python -m src.models.train")
    get_ipython().system("python -m src.models.evaluate")
    out = Path(f"reports/runs/{tag}_s{seed}")
    out.mkdir(parents=True, exist_ok=True)
    for f in ["metrics/train.json", "metrics/history.json", "metrics/test.json",
              "reports/preds_val.csv", "reports/preds_test.csv"]:
        shutil.copy(f, out)
    shutil.copy("checkpoints/best.pt", WORK / f"{tag}_s{seed}.pt")
    shutil.copytree(out, WORK / "runs" / out.name, dirs_exist_ok=True)  # sauvegarde si la session coupe
    set_seed(42)
    print(f"✔ {tag} (graine {seed}) rangé dans {out}")

## 4. Comparer les trois modèles
Chaque run apparaît dans MLflow (nom : `modèle-augmentation-sgraine`).

In [ ]:
run("baseline")      # Baseline CNN entraîné from scratch

In [ ]:
run("efficientnet")  # EfficientNet-V2-S (transfer learning ImageNet)

In [ ]:
run("vit")           # ViT-B/16 (Vision Transformer)

## 5. Variante `scale` du ViT : réduire la dépendance au cadrage
Le test de robustesse a montré qu'un simple recadrage fait basculer une image générée en « réel ».
On réentraîne le même ViT avec des recadrages aléatoires (50 à 100 % de l'image) pendant l'entraînement.

In [ ]:
run("vit_scale")

## 6. (Facultatif) Variabilité entre graines
Deux entraînements supplémentaires d'EfficientNet et du ViT (graines 1 et 2) pour donner les résultats en
moyenne ± écart-type. Saute cette section si le temps GPU manque : les intervalles de confiance bootstrap (section 8)
restent disponibles.

In [ ]:
for seed in (1, 2):
    run("efficientnet", seed)
    run("vit", seed)

## 7. Tableau comparatif

In [ ]:
import json, pandas as pd
rows = []
for d in sorted(Path("reports/runs").iterdir()):
    tr, te = json.load(open(d / "train.json")), json.load(open(d / "test.json"))
    rows.append({"run": d.name, **{k: tr[k] for k in ("val_accuracy", "val_f1", "val_roc_auc", "train_time_s", "n_params_m")},
                 **{k: te.get(k) for k in ("test_accuracy", "test_f1", "test_roc_auc", "test_errors", "external_accuracy")}})
pd.DataFrame(rows)

## 8. Analyses pour le rapport
- **Robustesse quantitative** sur 1 000 images de test (et non une seule), ViT `basic` contre ViT `scale`.
- **Statistiques** : intervalles de confiance bootstrap à 95 %, test de McNemar, variabilité entre graines.
- **Calibration** : ECE, diagramme de fiabilité, temperature scaling.
- **Courbes d'apprentissage**.
- **Intérieur du ViT** : embeddings de position, distance d'attention par couche (comparée aux poids ImageNet).

Chaque script écrit sa figure dans `docs/rapport/figures/` et son tableau dans `docs/rapport/generated/` :
le rapport LaTeX les inclut automatiquement.

In [ ]:
for tag, model in [("efficientnet", "efficientnet"), ("vit", "vit"), ("vit_scale", "vit")]:
    !python -m src.analysis.robustness --tag {tag} --model {model} --checkpoint /kaggle/working/{tag}_s42.pt
!python -m src.analysis.robustness --report

In [ ]:
!python -m src.analysis.stats
!python -m src.analysis.calibration
!python -m src.analysis.learning_curves
!python -m src.analysis.vit_internals --checkpoint /kaggle/working/vit_s42.pt --compare-pretrained

In [ ]:
from IPython.display import Image, display
for f in ["learning_curves", "robustness", "calibration", "vit_pos_embed", "vit_attention_distance"]:
    print(f); display(Image(f"reports/{f}.png"))

## 9. Évaluer, expliquer et exporter le modèle retenu
Choisis le modèle déployé (`MODEL_TAG`). Garde `vit` sauf si `vit_scale` a une exactitude de test équivalente
**et** une meilleure robustesse (section 8) : dans ce cas, déploie `vit_scale`.
L'export ONNX embarque l'explication : **CAM** pour un CNN, **Attention Rollout** pour le ViT.

In [ ]:
MODEL_TAG = "vit"   # baseline | efficientnet | vit | vit_scale

set_params(**HP[MODEL_TAG]); set_seed(42)
shutil.copy(WORK / f"{MODEL_TAG}_s42.pt", "checkpoints/best.pt")
for f in ("train.json", "history.json"):
    shutil.copy(f"reports/runs/{MODEL_TAG}_s42/{f}", f"metrics/{f}")
!python -m src.models.evaluate
!python -m src.utils.gradcam
!python -m src.models.export_onnx

In [ ]:
# Tout ce qu'il faut récupérer, dans un seul zip (onglet Output du notebook)
!zip -qr /kaggle/working/deepguard_outputs.zip checkpoints/best.pt data/processed models metrics reports params.yaml docs/rapport/generated docs/rapport/figures
!ls -lh /kaggle/working/deepguard_outputs.zip

## 10. De retour sur ton ordinateur
Dézippe `deepguard_outputs.zip` dans un dossier **séparé**, puis copie dans le dépôt : `checkpoints/`, `data/processed/`,
`models/`, `metrics/`, `reports/`, `docs/rapport/generated/` et `docs/rapport/figures/`. Mets dans `params.yaml` les
hyperparamètres du modèle retenu (pas les chemins Kaggle), puis :
```bash
dvc commit -f prepare train evaluate explain export
dvc add reports/runs          # prédictions et historiques de toutes les expériences (analyses reproductibles)
dvc push
git add dvc.lock params.yaml metrics reports/runs.dvc docs/rapport/generated docs/rapport/figures
git commit -m "feat: analyses (IC, McNemar, calibration, robustesse, ViT)" && git push
cd docs/rapport && pdflatex rapport_technique.tex && pdflatex rapport_technique.tex
```